In [1]:
import sys
import os
import pandas as pd
import numpy as np

sys.path.append('..')

import utilities.functions as functions

from utilities.functions import (
    #load_data,
    check_key_uniqueness,
    merge_df,
    load_orders,
    process_orders_pandas
)
from utilities.load_data import (

   load_data,
   count_null

)

In [2]:
from pathlib import Path
BASE_PATH = Path("/Users/maceli/ifood_cs/dados") 



Load the data --- se necessario salvar em stage - neste momento o estara comentado

In [3]:
URL_CONSUMER = "https://data-architect-test-source.s3-sa-east-1.amazonaws.com/consumer.csv.gz"
df_consumer = load_data(URL_CONSUMER)[["customer_id", "active","created_at"]]

In [4]:
URL_RESTAURANT ="https://data-architect-test-source.s3-sa-east-1.amazonaws.com/restaurant.csv.gz"
df_restaurant= load_data(URL_RESTAURANT)

In [5]:
ab_test_url = "https://data-architect-test-source.s3-sa-east-1.amazonaws.com/ab_test_ref.tar.gz"
df_ab = load_data(ab_test_url)

In [6]:
#save_parquet(df_consumer, "stage", "df_consumer.parquet")
#save_parquet(df_restaurant, "stage", "df_restaurant.parquet")
#save_parquet(df_consumer, "stage", "df_consumer.parquet")

Bronze layer - verify duplicates e nulo. e se necessario remover

In [7]:
check_key_uniqueness(df_consumer, ["customer_id","active","created_at"])

✅ Colunas ['customer_id', 'active', 'created_at'] são NOT NULL e UNIQUE.


(True, None, None, None)

In [8]:
check_key_uniqueness(df_restaurant, ["id"])

✅ Colunas ['id'] são NOT NULL e UNIQUE.


(True, None, None, None)

In [9]:
check_key_uniqueness(df_ab, ["customer_id","is_target"])
print(df_ab[df_ab["customer_id"].isna()])


❌ Colunas ['customer_id', 'is_target'] contêm valores nulos.

Soma de nulos por coluna:
customer_id    1
is_target      0
dtype: int64

Índices com nulos:
[81149]
      customer_id is_target
81149         NaN    target


Valida nullos

In [16]:
df_consumer_null=count_null(df_consumer)
df_consumer_null.head()

,Total,Percent
customer_id,0,0.0
active,0,0.0
created_at,0,0.0


In [15]:
df_restaurant_null=count_null(df_restaurant)
df_restaurant_null.head()

,Total,Percent
minimum_order_value,95,0.013028
delivery_time,1,0.000137
id,0,0.000000
created_at,0,0.000000
enabled,0,0.000000


In [17]:
df_ab_null=count_null(df_ab)
df_ab_null.head()

,Total,Percent
customer_id,1,0.000001
is_target,0,0.000000


In [18]:
df_ab_np = df_ab[~df_ab["customer_id"].isna()]

Antes de carregar a base ordens sera definido o publico todal, e da base de ordem serao filtrados somentes os clientes elegiceis

id=['fffe7b38b14ac2fca8906500fbecf87f5f8470179ed7ffa974892a3a15650604',
       'fffd6a19e4affba4589945ba2fe76804f25ad9301c0d3011219766b51106c2fe',
       'fffad85994233d99370bfb55ff196c7e82af11dd9825b121fa5f5563c2666c2a',
       '00086dd0b93a9c96d2d13e1b245bb82abbea957306b193be39375bdd853811f9',
       '00070a129efd4c5ffe3dfbc2ca704a7f891fd8b0ab4159930b813c541194c0cc',
       '0000c21984ae00cefb5d4931bfa49483dde546413c9b40c4228220f27d7ecdf2'
       ]
df_ab_np=df_ab_np[df_ab_np['customer_id'].isin(id)]

In [19]:
df_publico=merge_df(df_ab_np,df_consumer,['customer_id'],'left')

In [20]:
clientes_mes = (
    df_publico.groupby(['active', 'is_target'], dropna=False)
              ['customer_id']
              .nunique()
              .reset_index(name='numero_clientes_distintos')
)
clientes_mes

,active,is_target,numero_clientes_distintos
0,True,control,359700
1,True,target,444861
2,NaN,control,129
3,NaN,target,181
4,False,control,713
5,False,target,882


In [21]:
df_publico = df_publico.dropna(subset=['active'])

In [22]:
clientes_mes = (
    df_publico.groupby(['active', 'is_target'], dropna=False)
              ['customer_id']
              .nunique()
              .reset_index(name='numero_clientes_distintos')
)
clientes_mes

,active,is_target,numero_clientes_distintos
0,False,control,713
1,False,target,882
2,True,control,359700
3,True,target,444861


Publico definido e todos os clientes marcados no teste a/b e existentes na base de clientes

Da base de ordens serao filtrados todos os clientes com orden nos meses de de dezembro e janeiro

In [ ]:
URL_ORDERS = "https://data-architect-test-source.s3-sa-east-1.amazonaws.com/order.json.gz"


COLUMNS_TO_DROP = [
    'cpf','customer_name','delivery_address_city','delivery_address_country',
   'delivery_address_district','delivery_address_external_id',
    'delivery_address_latitude','delivery_address_longitude',

    'delivery_address_zip_code','items',
    'merchant_latitude','merchant_longitude','merchant_timezone',
    'order_scheduled','order_scheduled_date'
]

customer_ids = df_ab["customer_id"].astype(str).unique()

df_orders = load_orders(
    url=URL_ORDERS,
    customer_ids=customer_ids,
    columns_to_drop=COLUMNS_TO_DROP
)

df_orders.head()


In [ ]:
print(check_key_uniqueness(df_orders, ["customer_id","merchant_id","order_id"]))
print(check_key_uniqueness(df_orders, ["customer_id","merchant_id","order_id","order_created_at"]))

Add orders para a base de publico

In [ ]:
df_publico_orders=merge_df(df_publico,df_orders,['customer_id'],'inner')

In [ ]:
#df_publico_orders.to_parquet(BASE_PATH / "silver" / "df_publico_orders.parquet", index=False)

Construcao de chave unica, e sumarizacoes visao cliente

In [ ]:
df_publico=process_orders_pandas(df_publico_orders)

In [ ]:
df_publico_orders = pd.read_parquet(BASE_PATH / "gold" / "df_publico_orders.parquet")

Uma linha por cliente, com as variaveis necessarias

In [ ]:
df_pub_un = df_publico[["customer_id","is_target", "order_created_month", "num_pedidos_mes", "num_pedidos_hist",'total_amount_mes','ticket_medio']].drop_duplicates().reset_index(drop=True)
df_pub_un['pedidos_sum'] = np.where(
        df_pub_un['num_pedidos_mes'] > 10, '10+',
        df_pub_un['num_pedidos_mes'].astype(str)
    )
df_pub_un.head()

Salvar base visao cliente em gold layer

In [ ]:
#Cdf_pub_un.to_parquet(BASE_PATH / "gold" / "df_pub_un.parquet", index=False)

In [ ]:
df_stats_mes = df_pub_un.groupby(['pedidos_sum', 'order_created_month']).agg(
    total_clientes=('customer_id', 'nunique')
)

df_stats_mes['pct_total_mes'] = (
    df_stats_mes['total_clientes'] /
    df_stats_mes.groupby('order_created_month')['total_clientes'].transform('sum') * 100
).round(2)

df_stats_mes.head(20)


In [ ]:
df_pub_hist = df_publico[["customer_id","is_target",  "num_pedidos_hist"]].drop_duplicates().reset_index(drop=True)

In [ ]:
df_stats_hist = df_pub_hist.groupby(['num_pedidos_hist']).agg(
    total_clientes=('customer_id', 'nunique')  
).round(2)

df_stats_hist['pct_total'] = (df_stats_hist['total_clientes'] / df_stats_hist['total_clientes'].sum() * 100).round(2)

df_stats_hist.head(20)